# MCP schema-less арм: отдаём LoRA-адаптер через vLLM

Закрывает единственный незакрытый инженерный шаг ветки: MCP-сервер умеет режим
`with_schema=False` (знание о БД — в весах адаптера), но до сих пор демонстрировался
только `with_schema=True`, потому что базовая модель эндпоинта без адаптера галлюцинирует
имена таблиц. Здесь поднимается vLLM с нашим LoRA и отдаётся наружу, после чего локальный
MCP ходит в него как в обычный OpenAI-совместимый эндпоинт.

**Код MCP менять не нужно.** `main.py` собирает клиента из `LLM_BASE_URL` / `LLM_MODEL_NAME` /
`LLM_API_KEY`; достаточно переставить три значения в `.env`.

### Что нужно до запуска

1. **Internet: On** в настройках ноутбука (иначе не скачается ни vllm, ни базовая модель).
2. **Accelerator:** `GPU T4 x2` для 7B, любая одиночная GPU для 3B.
3. Папку адаптера залить как Kaggle Dataset — целиком, вместе с `adapter_config.json`
   и `adapter_model.safetensors` (в git лежат только карточки, веса gitignored).
4. Локально должен быть поднят ssh-туннель к БД — MCP исполняет SQL с вашей машины.

Схема: **БД и MCP локально, модель на Kaggle**. Наружу торчит только LLM.


## 1. Конфигурация


In [ ]:
# 3B — безопасный дефолт: влезает на одну T4 (16 ГБ) с запасом.
# 7B — победитель ветки; нужен ускоритель "GPU T4 x2" и TP_SIZE = 2.
BASE_MODEL  = "Qwen/Qwen2.5-Coder-3B-Instruct"   # 7B: "Qwen/Qwen2.5-Coder-7B-Instruct"
TP_SIZE     = 1                                   # 7B на T4 x2: 2

SERVED_NAME = "db2model"        # это же значение пойдёт в LLM_MODEL_NAME локального .env
PORT        = 8000
MAX_LEN     = 2048              # schema-less промпт короткий (~90 токенов), больше не нужно
GPU_UTIL    = 0.90

# ВНИМАНИЕ: туннель отдаёт сервер в публичный интернет. Без ключа его найдут и займут
# вашу GPU. Поменяйте на случайную строку и положите её же в LLM_API_KEY локально.
API_KEY     = "CHANGE-ME-db2model"

assert API_KEY != "CHANGE-ME-db2model", "Задайте свой API_KEY — сервер будет публичным"


## 2. Находим адаптер и сверяем его с базовой моделью

Классическая ошибка — подать 7B-адаптер к 3B-базе (или наоборот): vLLM стартует, а качество
молча падает до мусора. Здесь это ловится сразу.


In [ ]:
import glob, json, os

cands = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
assert cands, "Не найден adapter_config.json — залейте папку адаптера как Kaggle Dataset"
if len(cands) > 1:
    print("Найдено несколько адаптеров, беру первый:")
    for c in cands:
        print("  ", c)

ADAPTER_DIR = os.path.dirname(cands[0])
cfg = json.load(open(cands[0]))

print("адаптер :", ADAPTER_DIR)
print("база    :", cfg["base_model_name_or_path"])
print("r / alpha:", cfg["r"], "/", cfg["lora_alpha"])

assert cfg["base_model_name_or_path"] == BASE_MODEL, (
    f"Адаптер обучен на {cfg['base_model_name_or_path']}, а BASE_MODEL={BASE_MODEL}. "
    "Поправьте BASE_MODEL в ячейке конфигурации."
)
MAX_LORA_RANK = max(16, cfg["r"])
assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")), (
    "Рядом с adapter_config.json нет adapter_model.safetensors — залиты только карточки"
)
print("\nOK: адаптер и база совпадают")


## 3. Ставим vLLM

Минут 5–10. Ворнинги про несовместимость версий на Kaggle — обычное дело.


In [ ]:
!pip install -q -U vllm

import vllm
print("vllm", vllm.__version__)


## 4. Поднимаем сервер с LoRA

`--enable-lora` + `--lora-modules db2model=<путь>`: базовая модель грузится один раз, адаптер
прикладывается по имени модели в запросе. Запрос с `model="db2model"` идёт через адаптер,
с `model="base"` — мимо него, что удобно для контрольного сравнения.


In [ ]:
import subprocess, sys, time, os, requests

LOG = "/kaggle/working/vllm.log"
cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", BASE_MODEL,
    "--served-model-name", "base",
    "--enable-lora",
    "--lora-modules", f"{SERVED_NAME}={ADAPTER_DIR}",
    "--max-lora-rank", str(MAX_LORA_RANK),
    "--max-model-len", str(MAX_LEN),
    "--gpu-memory-utilization", str(GPU_UTIL),
    "--tensor-parallel-size", str(TP_SIZE),
    "--dtype", "half",
    "--port", str(PORT),
    "--api-key", API_KEY,
]
print(" ".join(cmd), "\n")

logf = open(LOG, "w")
server = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)

# Первый запуск качает веса — ждём до 15 минут.
deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        print(open(LOG).read()[-3000:])
        raise RuntimeError(f"vLLM упал, код {server.returncode} (лог выше)")
    try:
        if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
            print("\nсервер поднялся")
            break
    except requests.RequestException:
        pass
    time.sleep(10)
else:
    print(open(LOG).read()[-3000:])
    raise TimeoutError("vLLM не поднялся за 15 минут (лог выше)")

hdr = {"Authorization": f"Bearer {API_KEY}"}
print([m["id"] for m in requests.get(f"http://localhost:{PORT}/v1/models", headers=hdr).json()["data"]])


## 5. Проверяем адаптер локально, ещё до туннеля

Промпт — тот самый `SYSTEM_PROMPT_NOSCHEMA_TEMPLATE` из MCP: схема в него не подаётся.
Если модель называет реальные таблицы (`account`, `district`, `molecule`) — адаптер работает.
Для контраста тот же вопрос гоняется через `model="base"`, без адаптера.


In [ ]:
SYSTEM_NOSCHEMA = """
Ты экспертный Text2SQL ассистент. Ты уже обучен на конкретной базе данных и знаешь её
схему, таблицы и колонки — они НЕ приводятся в промпте, бери их из своих знаний.
Твоя задача - точно преобразовывать запросы на русском языке в оптимизированные SQL-запросы.

### Правила генерации SQL:
1. Всегда используй алиасы для таблиц в JOIN
2. Только SELECT запросы (никаких модификаций)
3. Диалект - PostgreSQL
"""

def ask(question, model):
    r = requests.post(
        f"http://localhost:{PORT}/v1/chat/completions",
        headers=hdr,
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": SYSTEM_NOSCHEMA},
                {"role": "user", "content": question},
            ],
            "temperature": 0.0,
            "max_tokens": 256,
        },
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()

QUESTIONS = [
    "Сколько всего счетов в базе?",
    "Перечисли уникальные типы связей между атомами.",
    "Сколько зарегистрировано пользователей?",
]

for q in QUESTIONS:
    print("=" * 70)
    print("ВОПРОС :", q)
    print("-- адаптер --\n", ask(q, SERVED_NAME))
    print("-- база без адаптера --\n", ask(q, "base"))


## 6. Отдаём наружу через cloudflared

Бесплатный `trycloudflare` — без регистрации, URL живёт, пока жив процесс.
Сервер закрыт ключом из конфигурации, но URL всё равно не публикуйте.


In [ ]:
import re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

TLOG = "/kaggle/working/tunnel.log"
tlogf = open(TLOG, "w")
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=tlogf, stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
deadline = time.time() + 120
while time.time() < deadline and PUBLIC_URL is None:
    time.sleep(3)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TLOG).read())
    if m:
        PUBLIC_URL = m.group(0)

assert PUBLIC_URL, "Не получил URL туннеля — смотрите /kaggle/working/tunnel.log"
print("URL:", PUBLIC_URL)

# Проверка снаружи. НЕ блокирующая: свежее имя trycloudflare нередко не резолвится
# собственным DNS Kaggle, хотя туннель при этом жив. Настоящая проверка — с вашей машины.
for attempt in range(20):
    try:
        ext = requests.get(f"{PUBLIC_URL}/v1/models", headers=hdr, timeout=15)
        ext.raise_for_status()
        print("туннель отвечает, видны модели:", [m["id"] for m in ext.json()["data"]])
        break
    except requests.RequestException as e:
        print(f"попытка {attempt + 1}/20: {type(e).__name__}, жду 10 с")
        time.sleep(10)
else:
    print("!! Изнутри Kaggle имя не резолвится — это почти всегда его DNS, а не туннель.")
    print("   Проверьте с локальной машины:")
    print(f'   curl -H "Authorization: Bearer {API_KEY}" {PUBLIC_URL}/v1/models')

print("\n" + "=" * 70)
print("Впишите в локальный .env:\n")
print(f"LLM_BASE_URL={PUBLIC_URL}/v1")
print(f"LLM_MODEL_NAME={SERVED_NAME}")
print(f"LLM_API_KEY={API_KEY}")
print("TEXT2SQL_WITH_SCHEMA=false")
print("=" * 70)


## 7. Что сделать локально

Ноутбук отсюда не закрывать — с ним умрут и сервер, и туннель.

```bash
uv run --env-file .env python db2model/mcp_smoke_test.py
```

Ожидаемое: `TEXT2SQL_WITH_SCHEMA=false`, схема в промпт не подаётся, SQL приходит от адаптера
и исполняется на живой БД через ваш ssh-туннель. Это и есть недостающая демонстрация —
тот самый арм, который защищается.

### Если что-то пошло не так

| Симптом | Причина |
|---|---|
| vLLM падает с OOM | 7B на одной T4 не влезает — либо `GPU T4 x2` + `TP_SIZE=2`, либо 3B |
| `model not found: db2model` | `--lora-modules` не подхватился, смотрите `/kaggle/working/vllm.log` |
| SQL с несуществующими таблицами | подан не тот адаптер к не той базе, либо запрос ушёл на `base` |
| туннель отвалился | trycloudflare рвётся; перезапустите ячейку 6, URL сменится |
| сессия Kaggle умерла | она живёт ~9 часов и рвётся при простое — для разовой демонстрации нормально, для сервиса нет |
